### **Topics:**
- tf.keras.Sequential()
- tf.keras.layers.Dense()
- history.history['loss'][0]
- model.compile()
- model.fit()
- EarlyStopping
- Tensorflow - functional API

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

In [ ]:
# Here are our inputs.
X = np.array([[1, 3, -2, 0],
              [1, 1, 0, 1]])
Y = np.array([0, 1])

# initialize W
W = [1, 1, 1, 1]

### **TensorFlow/Kears:**

In [ ]:
tf.keras.backend.clear_session # clear previous model from memory
model = tf.keras.Sequential()
model.add(
    tf.keras.layers.Dense(
        units=1,
        activation='sigmoid',
        use_bias=False,
        kernel_initializer=tf.ones_initializer
    )
)

optimizer = tf.keras.optimizers.SGD(learning_rate=0.1)
model.compile(loss='binary_crossentropy', optimizer=optimizer)

In [ ]:
preds = model.predict(X)

history = model.fit(
    X = X,
    y = Y,
    epochs=1, # number of times the learning algorithm will work through the entire training dataset.
    batch_size=2,
    verbose=0
)

In [ ]:
loss = history.history['loss'][0] # [0] refers to the epochs 1

In [ ]:
weights = model.layers[0].get_weights()[0].T # [0] refers to the epochs 1

### tf.keras.Sequential():

`tf.keras.Sequential()` initializes a linear stack of layers. It is the simplest way to build a model in Keras, where you add layers one by one in the order they should be processed.

#### Key Characteristics
* **Linear Topology:** Layers are connected sequentially. Data flows from the first layer to the last in a straight line.
* **No Layer Reuse:** You cannot use the same layer instance multiple times in different parts of the model (for that, you would use the **Functional API**).
* **No Multiple Inputs/Outputs:** It is designed for models with a single input tensor and a single output tensor.

#### Basic Usage Example
```python
import tensorflow as tf

model = tf.keras.Sequential([
    # Input layer (defines the shape of the incoming data)
    tf.keras.layers.Dense(64, activation='relu', input_shape=(10,)), 
    
    # Hidden layer
    tf.keras.layers.Dense(32, activation='relu'),
    
    # Output layer (e.g., for binary classification)
    tf.keras.layers.Dense(1, activation='sigmoid')
])
```

#### Common Methods
* **`.add()`**: Adds a layer to the stack after the model has been initialized.
* **`.compile()`**: Configures the model for training (specifies optimizer, loss function, and metrics).
* **`.fit()`**: Trains the model on a dataset.
* **`.evaluate()`**: Tests the model performance on a validation/test set.
* **`.predict()`**: Generates output predictions for new input data.
* **`.summary()`**: Prints a text summary of the model's architecture, layer types, and parameter counts.
#
---

### tf.keras.layers.Dense：

`tf.keras.layers.Dense` implements a **fully connected** neural network layer. In this layer, every input neuron is connected to every output neuron.

#### Mathematical Operation
For an input vector $x$, the layer performs the following transformation:
$$\text{output} = \text{activation}(x \cdot W + b)$$

Where:
* **$x$**: The input tensor.
* **$W$**: The weight matrix (learnable parameters).
* **$b$**: The bias vector (learnable parameter).
* **$\text{activation}$**: A non-linear function applied to the result.

#### Key Arguments
* **`units`**: The dimensionality of the output space (number of neurons in the layer).
* **`activation`**: The activation function to use (e.g., `'relu'`, `'sigmoid'`, `'softmax'`, `'tanh'`, or `None`).
* **`use_bias`**: Boolean, whether to use a bias vector. Defaults to `True`.
* **`kernel_initializer`**: Initializer for the weights (e.g., `'glorot_uniform'`, `'he_normal'`).
* **`bias_initializer`**: Initializer for the biases (e.g., `'zeros'`).

#### Example Usage
```python
import tensorflow as tf

# A Dense layer with 32 neurons and ReLU activation
layer = tf.keras.layers.Dense(units=32, activation='relu')

# Example input (batch_size=1, input_dim=10)
input_data = tf.random.normal([1, 10])

# Passing data through the layer
output = layer(input_data)

print(output.shape) # Output: (1, 32)
```

#### Common Use Cases
1.  **Hidden Layers:** Using `'relu'` to learn complex non-linear patterns.
2.  **Output Layer (Binary Classification):** Using `units=1` and `'sigmoid'` to output a probability.
3.  **Output Layer (Multi-class Classification):** Using `units=N` (where N is the number of classes) and `'softmax'` to output a probability distribution.
4.  **Output Layer (Regression):** Using `units=1` and no activation (or `'linear'`) to predict a continuous value.
#
---

### history.history['loss'][0]:

The `[0]` is used to access the **first element** in the list of loss values recorded during training.

When you call `model.fit()`, Keras returns a `History` object. The `history.history` attribute is a **dictionary** where the keys are the names of the metrics you tracked (like `'loss'`, `'accuracy'`, etc.), and the values are **lists** containing the value of that metric at the end of every epoch.

##### Breakdown of the structure:

If you train for **3 epochs**, `history.history['loss']` will look like this:
`[0.65, 0.42, 0.31]`

* `history.history['loss'][0]` $\rightarrow$ The loss value after **Epoch 1**.
* `history.history['loss'][1]` $\rightarrow$ The loss value after **Epoch 2**.
* `history.history['loss'][2]` $\rightarrow$ The loss value after **Epoch 3**.

##### Why use `[0]` in your specific code?
In your previous snippet, you set **`epochs=1`**:
```python
epochs=1
```
Because you only ran the training for one single epoch, the list `history.history['loss']` only contains **one value**. To extract that single numerical value from the list so you can print it or use it in a calculation, you must access it by its index, which is `0`.

#### history.history:
In Keras/TensorFlow, the `history.history` dictionary contains the metrics specified during the `model.compile()` step. Depending on your configuration, the available features typically include:

*   **`loss`**: The value of the loss function for the current epoch.
*   **`accuracy`**: The training accuracy (if `metrics=['accuracy']` was used).
*   **`mae`**: Mean Absolute Error (if `metrics=['mae']` was used).
*   **`mse`**: Mean Squared Error (if `metrics=['mse']` was used).
*   **`val_loss`**: The loss value on the validation data (if `validation_data` was provided).
*   **`val_accuracy`**: The validation accuracy (if `validation_data` was provided).
*   **`val_mae`**: Validation Mean Absolute Error.
*   **`val_mse`**: Validation Mean Squared Error.

#
---

### **model.compile():**

`model.compile()` is the step where you configure the learning process of a neural network before training begins. It defines how the model should interpret the error and how it should update its internal weights to minimize that error.

It requires three primary arguments:

1.  **Optimizer**: The algorithm used to update the weights of the network to reduce the loss. Common examples include `'adam'`, `'sgd'` (Stochastic Gradient Descent), and `'rmsprop'`.
2.  **Loss Function**: A mathematical formula that measures how far the model's predictions are from the actual targets. Common examples include `'mean_squared_error'` (for regression) and `'categorical_crossentropy'` (for multi-class classification).
3.  **Metrics**: Used to monitor the performance of the model during training and testing. These are not used for training the model itself, but for human evaluation. Common examples include `['accuracy']` or `['mae']`.

**Example usage:**
```python
model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['accuracy']
)
```
#
---

### **model.fit():**

`model.fit()` is the method used to train the neural network. It executes the actual training process by passing the input data through the model, calculating the error, and updating the weights using the optimizer defined in `model.compile()`.

#### Key Arguments

*   **`x`**: The input data (features) used for training.
*   **`y`**: The target data (labels/ground truth) the model is trying to predict.
*   **`epochs`**: The number of times the entire dataset is passed through the model. One epoch means every sample in the dataset has had an opportunity to update the model's weights once.
*   **`batch_size`**: The number of samples processed before the model's internal parameters are updated. For example, if you have 1,000 samples and a `batch_size` of 10, the model will perform 100 updates per epoch.
*   **`validation_data`**: A tuple `(x_val, y_val)` used to evaluate the model's performance on unseen data at the end of each epoch.
*   **`validation_split`**: A fraction of the training data (e.g., `0.2`) to be set aside as validation data, so you don't need to provide a separate dataset.
*   **`callbacks`**: A list of functions to be applied at certain stages during training (e.g., `EarlyStopping` to stop training when the loss stops improving, or `TensorBoard` for visualization).
*   **`verbose`**: Controls the logging output.
    *   `0`: Silent (no output).
    *   `1`: Progress bar (shows progress for each epoch).
    *   `2`: One line per epoch.

#### The Training Loop (Under the Hood)
When you call `model.fit()`, the following cycle repeats for every epoch and every batch:
1.  **Forward Pass**: The model makes predictions based on the current weights.
2.  **Loss Calculation**: The loss function measures the difference between predictions and actual targets.
3.  **Backward Pass (Backpropagation)**: The gradient of the loss is calculated with respect to the weights.
4.  **Weight Update**: The optimizer adjusts the weights in the direction that reduces the loss.
#
---

### EarlyStopping:

`EarlyStopping` is a callback used to prevent **overfitting** by stopping the training process as soon as the model's performance on a validation set stops improving.

### Key Parameters

*   **`monitor`**: The metric to watch. 
    *   Common values: `'val_loss'` (most common) or `'val_accuracy'`.
*   **`patience`**: The number of epochs to wait for an improvement before stopping. 
    *   If `patience=5`, and the loss doesn't improve for 5 consecutive epochs, training terminates.
*   **`min_delta`**: The minimum change in the monitored metric to qualify as an improvement. 
    *   If `min_delta=0.001`, a decrease in loss of `0.0001` will not be considered an improvement.
*   **`mode`**: Determines whether "improvement" means the metric is increasing or decreasing.
    *   `'min'`: Training stops when the metric stops **decreasing** (used for `loss`).
    *   `'max'`: Training stops when the metric stops **increasing** (used for `accuracy`).
    *   `'auto'`: Automatically infers based on the metric name.
*   **`restore_best_weights`**: (Crucial) If `True`, the model's weights are reverted to the state they were in during the last epoch where the monitored metric was at its best. If `False`, the model keeps the weights from the final (potentially overfitted) epoch.

### Practical Example

```python
from tensorflow.keras.callbacks import EarlyStopping

# Define the callback
early_stopping = EarlyStopping(
    monitor='val_loss',    # Watch validation loss
    patience=10,           # Wait 10 epochs for improvement
    min_delta=0.001,       # Improvement must be at least 0.001
    mode='min',            # We want to minimize loss
    restore_best_weights=True # Keep the weights from the best epoch
)

# Use it in model.fit
model.fit(
    X_train, y_train,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping]
)
```

### Why use it?
1.  **Prevents Overfitting**: Stops the model from learning noise in the training data once the validation error starts rising.
2.  **Saves Time/Resources**: Prevents wasting computational power on epochs that no longer improve the model.
3.  **Automates Hyperparameter Tuning**: You can set a high number of `epochs` (e.g., 1000) and let `EarlyStopping` decide exactly when the model has reached its peak performance.
#
---

### **Functunal API:**

The Keras Functional API is a flexible way to define models by treating layers as functions that take tensors as inputs and return tensors as outputs. Unlike the `Sequential` API, which is limited to a linear stack of layers, the Functional API allows for complex architectures.

#### Key Capabilities
*   **Multiple Inputs/Outputs:** Models can accept multiple input tensors and produce multiple output tensors.
*   **Non-linear Topology:** Supports residual connections (skip connections), branching, and merging (e.g., concatenation or addition).
*   **Shared Layers:** Allows a single layer instance to be reused multiple times within the same model.
*   **Non-sequential Data Flow:** Enables complex architectures like Siamese networks or Inception modules.

#### Basic Implementation Pattern
```python
from tensorflow.keras import layers, models, Input

# 1. Define the input shape and type
inputs = Input(shape=(784,))

# 2. Call layers as functions on the input/tensors
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dense(64, activation='relu')(x)

# 3. Define the output
outputs = layers.Dense(10, activation='softmax')(x)

# 4. Instantiate the model by specifying inputs and outputs
model = models.Model(model=inputs, outputs=outputs)
```

#### Complex Architecture Example (Residual Connection)
```python
inputs = Input(shape=(64,))
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dense(64, activation='relu')(x)

# Skip connection: adding the original input to the processed tensor
x = layers.add([x, inputs]) 

outputs = layers.Dense(10, activation='softmax')(x)
model = models.Model(inputs=inputs, outputs=outputs)
```

#### Comparison Summary
| Feature | Sequential API | Functional API |
| :--- | :--- | :--- |
| **Complexity** | Simple, linear stacks | Complex, non-linear graphs |
| **Input/Output** | Single input, single output | Multiple inputs, multiple outputs |
| **Flexibility** | Low | High |
| **Use Case** | Standard MLPs, simple CNNs | ResNets, Inception, Siamese Nets |